In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================================
# 0. FILE PATHS
# ============================================================
RAW_FILE = "Phenolics_RawData.xlsx"
OUT_CONC_FILE = "Phenolics_concentrations.csv"

# Optional:
LONGTERM_YIELD_FILE = "cleaned_data/longterm_yield_analysis_locked.csv"
ACUTE_YIELD_FILE = "cleaned_data/acute_yield_analysis_locked.csv"

# ============================================================
# 1. LOAD RAW PHENOLICS DATA
# ============================================================
ph = pd.read_excel(RAW_FILE)

# Standardize column names
ph.columns = (
    ph.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Columns:")
print(ph.columns.tolist())

# Expected names after cleaning are usually something close to:
# date, sample_id, tube, rep1_absorbance_765_nm, rep2_absorbance_765_nm, rep3_absorbance_765_nm
# If "tube#" became something else, fix it below.

# Try to identify key columns robustly
def find_col(possible_names, cols):
    for name in possible_names:
        if name in cols:
            return name
    return None

date_col = find_col(["date"], ph.columns)
sample_col = find_col(["sample_id", "sample"], ph.columns)
tube_col = find_col(["tube", "tube_"], ph.columns)

rep1_col = find_col(["rep1_absorbance_765_nm", "rep1_absorbance_765nm", "rep1"], ph.columns)
rep2_col = find_col(["rep2_absorbance_765_nm", "rep2_absorbance_765nm", "rep2"], ph.columns)
rep3_col = find_col(["rep3_absorbance_765_nm", "rep3_absorbance_765nm", "rep3"], ph.columns)

required = [date_col, sample_col, tube_col, rep1_col, rep2_col, rep3_col]
if any(x is None for x in required):
    raise ValueError(
        f"Could not identify all required columns.\n"
        f"Detected: date={date_col}, sample={sample_col}, tube={tube_col}, "
        f"rep1={rep1_col}, rep2={rep2_col}, rep3={rep3_col}"
    )

# Keep only needed columns and rename to a clean standard
ph = ph[[date_col, sample_col, tube_col, rep1_col, rep2_col, rep3_col]].copy()
ph.columns = ["date", "sample_id", "tube", "rep1_abs", "rep2_abs", "rep3_abs"]

# Clean values
ph["sample_id"] = ph["sample_id"].astype(str).str.strip()
ph["sample_id_upper"] = ph["sample_id"].str.upper()
ph["date"] = pd.to_datetime(ph["date"], errors="coerce")

for c in ["tube", "rep1_abs", "rep2_abs", "rep3_abs"]:
    ph[c] = pd.to_numeric(ph[c], errors="coerce")

# Standards are rows whose sample_id begins with Std/std
ph["is_standard"] = ph["sample_id_upper"].str.startswith("STD")

print("\nRaw phenolics shape:", ph.shape)
print("Number of standard rows:", ph["is_standard"].sum())
print("Number of non-standard rows:", (~ph["is_standard"]).sum())

# ============================================================
# 2. FIT CALIBRATION LINES BY (DATE, REP)
#    Requirement: separate regression line for each day and rep
# ============================================================
rep_map = {
    "rep1_abs": "concentration_rep1",
    "rep2_abs": "concentration_rep2",
    "rep3_abs": "concentration_rep3",
}

# Output frame starts with non-standards only, same wide format except last 3 cols are concentrations
samples = ph.loc[~ph["is_standard"], ["date", "sample_id", "tube", "rep1_abs", "rep2_abs", "rep3_abs"]].copy()
samples["concentration_rep1"] = np.nan
samples["concentration_rep2"] = np.nan
samples["concentration_rep3"] = np.nan

calibration_rows = []

for date_value, g in ph.groupby("date", dropna=False):
    std = g[g["is_standard"]].copy()
    samp_idx = samples["date"].eq(date_value)

    # Use tube as known standard concentration per assignment
    std = std.dropna(subset=["tube"])

    for rep_abs_col, out_col in rep_map.items():
        std_rep = std.dropna(subset=[rep_abs_col]).copy()

        # Need at least 2 points to fit line, but realistically should be many
        if len(std_rep) < 2:
            print(f"Skipping calibration for date={date_value}, {rep_abs_col}: not enough standard points")
            continue

        x = std_rep["tube"].astype(float).values
        y = std_rep[rep_abs_col].astype(float).values

        # Fit y = m*x + b
        m, b = np.polyfit(x, y, 1)

        # Compute R^2 for tracking
        y_hat = m * x + b
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot

        calibration_rows.append({
            "date": date_value,
            "replicate": rep_abs_col,
            "slope_m": m,
            "intercept_b": b,
            "n_standards": len(std_rep),
            "r2": r2
        })

        # Apply to non-standards from same date:
        # x = (y - b) / m, then multiply by 10 because juice was diluted 10:1
        use = samp_idx & samples[rep_abs_col].notna()
        samples.loc[use, out_col] = ((samples.loc[use, rep_abs_col] - b) / m) * 10

calibration_df = pd.DataFrame(calibration_rows)

print("\nCalibration summary:")
print(calibration_df.head())
print("\nNumber of date-rep calibrations fit:", len(calibration_df))

# ============================================================
# 3. SAVE REQUIRED CSV
#    Required file should contain only non-standard tubes and the 3 concentration columns
# ============================================================
phenolics_conc = samples[["date", "sample_id", "tube", "concentration_rep1", "concentration_rep2", "concentration_rep3"]].copy()

phenolics_conc.to_csv(OUT_CONC_FILE, index=False)
print(f"\nSaved required file: {OUT_CONC_FILE}")

# ============================================================
# 4. CREATE A MEAN CONCENTRATION FOR ANALYSIS
# ============================================================
phenolics_conc["mean_concentration"] = phenolics_conc[
    ["concentration_rep1", "concentration_rep2", "concentration_rep3"]
].mean(axis=1, skipna=True)

# ============================================================
# 5. PARSE SAMPLE IDs
#    Assignment notes:
#    - ST / MQ = cultivar
#    - LT + plot number = long-term
#    - Acute examples:
#        B3   = treatment B, replicate 3
#        BC2  = control for treatment B, replicate 2
#        A0   should be ignored
# ============================================================
def parse_sample_id(sample_id):
    s = str(sample_id).strip().upper()

    out = {
        "raw_sample_id": s,
        "cultivar": None,
        "experiment": None,
        "treatment": None,
        "is_control": None,
        "plot": None,
        "replicate": None,
        "pair_set": None,
        "pulse_num": None,
        "exclude_a0": False,
    }

    # cultivar
    if s.startswith("ST"):
        out["cultivar"] = "St"
        rest = s[2:]
    elif s.startswith("MQ"):
        out["cultivar"] = "MQ"
        rest = s[2:]
    else:
        return out

    # long-term: LT followed by plot number
    m_lt = re.fullmatch(r"LT(\d+)", rest)
    if m_lt:
        plot = int(m_lt.group(1))
        out["experiment"] = "long_term"
        out["plot"] = plot
        out["is_control"] = plot in [5, 6, 7, 8, 13, 14, 15, 16]
        out["treatment"] = "Control" if out["is_control"] else "OTC"

        # set mapping from assignment design
        if 1 <= plot <= 4:
            pair_set = 6 + plot      # 1->7, 2->8, 3->9, 4->10
        elif 5 <= plot <= 8:
            pair_set = plot + 2      # 5->7, 6->8, 7->9, 8->10
        elif 9 <= plot <= 12:
            pair_set = plot + 2      # 9->11,10->12,11->13,12->14
        elif 13 <= plot <= 16:
            pair_set = plot - 2      # 13->11,14->12,15->13,16->14
        else:
            pair_set = None

        out["pair_set"] = pair_set
        return out

    # acute cases after cultivar prefix:
    # A03  -> treatment A0 replicate 3
    # A0C1 -> control for A0 replicate 1
    # B3   -> treatment B replicate 3
    # BC2  -> control for B replicate 2
    # C2   -> treatment C replicate 2
    # CC1  -> control for C replicate 1
    # D3   -> treatment D replicate 3
    #
    # Control acute pattern: ([A-D]0?)C([1-3])
    m_ctrl = re.fullmatch(r"([A-D]0?)C([1-3])", rest)
    if m_ctrl:
        trt = m_ctrl.group(1)
        rep = int(m_ctrl.group(2))
        out["experiment"] = "acute"
        out["treatment"] = trt
        out["is_control"] = True
        out["replicate"] = rep
        out["exclude_a0"] = (trt == "A0")
        out["pulse_num"] = {"A": 4, "B": 3, "C": 2, "D": 1}.get(trt, np.nan)
        return out

    # Non-control acute pattern: ([A-D]0?)([1-3])
    m_otc = re.fullmatch(r"([A-D]0?)([1-3])", rest)
    if m_otc:
        trt = m_otc.group(1)
        rep = int(m_otc.group(2))
        out["experiment"] = "acute"
        out["treatment"] = trt
        out["is_control"] = False
        out["replicate"] = rep
        out["exclude_a0"] = (trt == "A0")
        out["pulse_num"] = {"A": 4, "B": 3, "C": 2, "D": 1}.get(trt, np.nan)
        return out

    return out


parsed = phenolics_conc["sample_id"].apply(parse_sample_id).apply(pd.Series)
phen = pd.concat([phenolics_conc, parsed.drop(columns=["raw_sample_id"])], axis=1)

print("\nParsed sample ID summary:")
print(phen[["sample_id", "cultivar", "experiment", "treatment", "is_control", "plot", "replicate"]].head(15))

# Exclude A0, per assignment
phen = phen.loc[~phen["exclude_a0"]].copy()

# ============================================================
# 6. BASIC QA CHECKS
# ============================================================
print("\nCounts by experiment:")
print(phen["experiment"].value_counts(dropna=False))

print("\nCounts by treatment:")
print(phen["treatment"].value_counts(dropna=False))

print("\nMissing mean concentration:", phen["mean_concentration"].isna().sum())

# Optional: clip clearly impossible negative estimates if desired.
# Assignment does not explicitly say to clip them, so I leave them as-is by default.
# If you want to set negatives to NaN, uncomment below:
# for col in ["concentration_rep1", "concentration_rep2", "concentration_rep3", "mean_concentration"]:
#     phen.loc[phen[col] < 0, col] = np.nan

# ============================================================
# 7. LONG-TERM PHENOLICS ANALYSIS
# ============================================================
lt_phen = phen.loc[phen["experiment"] == "long_term"].copy()
lt_phen["cultivar"] = lt_phen["cultivar"].astype("category")
lt_phen["treatment"] = lt_phen["treatment"].astype("category")
lt_phen["pair_set"] = lt_phen["pair_set"].astype("category")

print("\nLong-term phenolics rows:", len(lt_phen))
print(lt_phen[["sample_id", "cultivar", "treatment", "plot", "pair_set", "mean_concentration"]].head())

if len(lt_phen) > 0 and lt_phen["mean_concentration"].notna().sum() > 0:
    # Use pair_set to respect long-term paired design
    m_lt = smf.ols(
        "mean_concentration ~ C(treatment) * C(cultivar) + C(pair_set)",
        data=lt_phen
    ).fit()

    print("\nLONG-TERM MODEL SUMMARY")
    print(m_lt.summary())

    # Raw group means for interpretation
    lt_means = (
        lt_phen.groupby(["cultivar", "treatment"], observed=True)["mean_concentration"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    print("\nLong-term phenolics means:")
    print(lt_means)

# ============================================================
# 8. ACUTE PHENOLICS ANALYSIS
# ============================================================
acute_phen = phen.loc[phen["experiment"] == "acute"].copy()
acute_phen["cultivar"] = acute_phen["cultivar"].astype("category")
acute_phen["treatment"] = acute_phen["treatment"].astype("category")
acute_phen["is_control"] = acute_phen["is_control"].astype(int)

# Useful label like A_OTC / A_Control
acute_phen["treatment_group"] = np.where(
    acute_phen["is_control"] == 1,
    acute_phen["treatment"] + "_Control",
    acute_phen["treatment"] + "_OTC"
)

print("\nAcute phenolics rows:", len(acute_phen))
print(acute_phen[["sample_id", "cultivar", "treatment", "is_control", "replicate", "mean_concentration"]].head())

if len(acute_phen) > 0 and acute_phen["mean_concentration"].notna().sum() > 0:
    # Categorical treatment-group model
    m_acute_cat = smf.ols(
        "mean_concentration ~ C(treatment_group) * C(cultivar)",
        data=acute_phen
    ).fit()

    print("\nACUTE MODEL SUMMARY (categorical treatment groups)")
    print(m_acute_cat.summary())

    acute_means = (
        acute_phen.groupby(["cultivar", "treatment_group"], observed=True)["mean_concentration"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    print("\nAcute phenolics means:")
    print(acute_means)

    # Dose-style model using pulse count, among OTC only
    acute_otc = acute_phen.loc[acute_phen["is_control"] == 0].copy()
    acute_otc = acute_otc.loc[acute_otc["pulse_num"].notna()].copy()

    if len(acute_otc) > 0:
        m_acute_dose = smf.ols(
            "mean_concentration ~ pulse_num * C(cultivar)",
            data=acute_otc
        ).fit()

        print("\nACUTE MODEL SUMMARY (dose-response, OTC only)")
        print(m_acute_dose.summary())

# ============================================================
# 9. OPTIONAL: MERGE WITH YIELD DATA IF THOSE FILES EXIST
# ============================================================
def try_read_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None

lt_yield = try_read_csv(LONGTERM_YIELD_FILE)
acute_yield = try_read_csv(ACUTE_YIELD_FILE)

if lt_yield is not None:
    print("\nLoaded long-term yield file for optional merge:", LONGTERM_YIELD_FILE)
    lt_yield.columns = (
        lt_yield.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    # Try common plot column names
    lt_plot_col = None
    for c in ["plot", "plot_number"]:
        if c in lt_yield.columns:
            lt_plot_col = c
            break

    if lt_plot_col is not None:
        lt_merge = lt_phen.merge(
            lt_yield,
            left_on="plot",
            right_on=lt_plot_col,
            how="left",
            suffixes=("", "_yield")
        )
        print("Merged long-term phenolics + yield rows:", len(lt_merge))

if acute_yield is not None:
    print("\nLoaded acute yield file for optional merge:", ACUTE_YIELD_FILE)
    acute_yield.columns = (
        acute_yield.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    # Expected keys vary, so this is only a starting point
    possible_trt = [c for c in acute_yield.columns if "treat" in c]
    possible_rep = [c for c in acute_yield.columns if "rep" in c]
    print("Potential acute yield treatment cols:", possible_trt)
    print("Potential acute yield replicate cols:", possible_rep)

# ============================================================
# 10. DIAGNOSTIC PLOTS
# ============================================================
def diag_plots(model, title_prefix="Model"):
    resid = model.resid
    fitted = model.fittedvalues

    plt.figure(figsize=(6, 4))
    plt.scatter(fitted, resid)
    plt.axh